Anime Recommendation System

In [9]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("CooperUnion/anime-recommendations-database")

print("Path to dataset files:", path)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: /Users/julieannsalangsang/.cache/kagglehub/datasets/CooperUnion/anime-recommendations-database/versions/1


In [10]:
import pandas as pd
df = pd.read_csv(f"{path}/anime.csv")

print(df.head())

print("\nDataset Information:" )
print(df.info())

print("\nMissing Values:")
print(df.isnull().sum())

print(df.describe())

print(df["episodes"].unique()[:30]) 

   anime_id                              name  \
0     32281                    Kimi no Na wa.   
1      5114  Fullmetal Alchemist: Brotherhood   
2     28977                          Gintama°   
3      9253                       Steins;Gate   
4      9969                     Gintama&#039;   

                                               genre   type episodes  rating  \
0               Drama, Romance, School, Supernatural  Movie        1    9.37   
1  Action, Adventure, Drama, Fantasy, Magic, Mili...     TV       64    9.26   
2  Action, Comedy, Historical, Parody, Samurai, S...     TV       51    9.25   
3                                   Sci-Fi, Thriller     TV       24    9.17   
4  Action, Comedy, Historical, Parody, Samurai, S...     TV       51    9.16   

   members  
0   200630  
1   793665  
2   114262  
3   673572  
4   151266  

Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Cou

Data Cleaning

In [11]:
df["episodes"] = pd.to_numeric(
    df["episodes"],
    errors="coerce"
)

print(df.isnull().sum())

df = df.dropna(
    subset=["name", "genre", "type", "rating"]
)

df = df.reset_index(drop=True)

print("Cleaned Dataset Shape:")
print(df.shape)

print("\nMissing Values")
print(df.isnull().sum())


anime_id      0
name          0
genre        62
type         25
episodes    340
rating      230
members       0
dtype: int64
Cleaned Dataset Shape:
(12017, 7)

Missing Values
anime_id      0
name          0
genre         0
type          0
episodes    187
rating        0
members       0
dtype: int64


Prepare the features. Comparing the anime based on there genre and type

genre + type --> similarity feature --> find anime with similar content

In [16]:
recommend_df = df[[
    "anime_id", 
    "name",
    "genre",
    "type",
    "rating",
    "members"
]].copy()

print(recommend_df.head())
print(recommend_df.info())

   anime_id                              name  \
0     32281                    Kimi no Na wa.   
1      5114  Fullmetal Alchemist: Brotherhood   
2     28977                          Gintama°   
3      9253                       Steins;Gate   
4      9969                     Gintama&#039;   

                                               genre   type  rating  members  
0               Drama, Romance, School, Supernatural  Movie    9.37   200630  
1  Action, Adventure, Drama, Fantasy, Magic, Mili...     TV    9.26   793665  
2  Action, Comedy, Historical, Parody, Samurai, S...     TV    9.25   114262  
3                                   Sci-Fi, Thriller     TV    9.17   673572  
4  Action, Comedy, Historical, Parody, Samurai, S...     TV    9.16   151266  
<class 'pandas.DataFrame'>
RangeIndex: 12017 entries, 0 to 12016
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12017 non-null  int64  
 1   name      12

In [17]:
# combined feature column
recommend_df["features"] = (
    recommend_df["genre"]+ " " + 
    recommend_df["type"]
)

print(
    recommend_df[
        ["features", "genre", "type", "features"]
    ].head()
)

                                            features  \
0         Drama, Romance, School, Supernatural Movie   
1  Action, Adventure, Drama, Fantasy, Magic, Mili...   
2  Action, Comedy, Historical, Parody, Samurai, S...   
3                                Sci-Fi, Thriller TV   
4  Action, Comedy, Historical, Parody, Samurai, S...   

                                               genre   type  \
0               Drama, Romance, School, Supernatural  Movie   
1  Action, Adventure, Drama, Fantasy, Magic, Mili...     TV   
2  Action, Comedy, Historical, Parody, Samurai, S...     TV   
3                                   Sci-Fi, Thriller     TV   
4  Action, Comedy, Historical, Parody, Samurai, S...     TV   

                                            features  
0         Drama, Romance, School, Supernatural Movie  
1  Action, Adventure, Drama, Fantasy, Magic, Mili...  
2  Action, Comedy, Historical, Parody, Samurai, S...  
3                                Sci-Fi, Thriller TV  
4  Action

Convert the text features into numbers

In [21]:
# converts each anime into numerical vector based on the words in the features column 

from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()

feature_matrix = vectorizer.fit_transform(
    recommend_df["features"]
)

print(feature_matrix.shape)

# check all the generated features in the feature names
print(vectorizer.get_feature_names_out())

(12017, 52)
['action' 'adventure' 'ai' 'arts' 'cars' 'comedy' 'dementia' 'demons'
 'drama' 'ecchi' 'fantasy' 'fi' 'game' 'harem' 'hentai' 'historical'
 'horror' 'josei' 'kids' 'life' 'magic' 'martial' 'mecha' 'military'
 'movie' 'music' 'mystery' 'of' 'ona' 'ova' 'parody' 'police' 'power'
 'psychological' 'romance' 'samurai' 'school' 'sci' 'seinen' 'shoujo'
 'shounen' 'slice' 'space' 'special' 'sports' 'super' 'supernatural'
 'thriller' 'tv' 'vampire' 'yaoi' 'yuri']
